In [ ]:
# import necessary libraries
import pandas as pd

In [ ]:
# read the two files containing match data and summoner's rank
matchDF = pd.read_csv('../aggregateMatchAndTimeline.csv')
print(matchDF.columns)

summonerRankDF = pd.read_csv('../summonerRanks.csv')
print(summonerRankDF.columns)

In [ ]:
matchWithSummonerRankDF = matchDF.merge(summonerRankDF, on='summonerId', how='right')
matchWithSummonerRankDF.head(20)

In [ ]:
matchPlayerCounts = matchWithSummonerRankDF.groupby(['gameId'])['summonerId'].count().reset_index()
matchPlayerCounts.rename(columns={'summonerId': 'playerCounts'}, inplace=True)
print(matchPlayerCounts.head())


matchWithSummonerRankDF = matchWithSummonerRankDF.merge(matchPlayerCounts, on='gameId', how='inner')

valid_matches = matchWithSummonerRankDF[matchWithSummonerRankDF['playerCounts'] == 10]

In [ ]:
# calculation of numerical rank of each individual player and the average rank in the play

# join the tier and rank to yield a combined value
valid_matches['tierRank'] = valid_matches['tier'] + '-' + valid_matches['rank']

rankedDict = {
    'NA':0,
    'IRON-IV':1,
    'IRON-III':2,
    'IRON-II':3,
    'IRON-I':4,
    'BRONZE-IV':5,
    'BRONZE-III':6,
    'BRONZE-II':7,
    'BRONZE-I':8,
    'SILVER-IV':9,
    'SILVER-III':10,
    'SILVER-II':11,
    'SILVER-I':12,
    'GOLD-IV':13,
    'GOLD-III':14,
    'GOLD-II':15,
    'GOLD-I':16,
    'PLATINUM-IV':17,
    'PLATINUM-III':18,
    'PLATINUM-II':19,
    'PLATINUM-I':20,
    'EMERALD-IV':21,
    'EMERALD-III':22,
    'EMERALD-II':23,
    'EMERALD-I':24,
    'DIAMOND-IV':25,
    'DIAMOND-III':26,
    'DIAMOND-II':27,
    'DIAMOND-I':28,
    'MASTER-I':29,
    'GRANDMASTER-I':30,
    'CHALLENGER-I':31
}

def calculateNumericalRank(row):
    return rankedDict[row['tierRank']]

valid_matches['numericalRank'] = valid_matches.apply(calculateNumericalRank, axis=1)

averageRankDF = valid_matches.groupby(['gameId'])['numericalRank'].mean().reset_index()
averageRankDF.rename(columns={'numericalRank': 'avgrank'}, inplace=True)

valid_matches = valid_matches.merge(averageRankDF, on='gameId', how='inner')

print(valid_matches.head())

valid_matches.sort_values(by=['gameId'], inplace=True)

In [ ]:
# export to csv
valid_matches.to_csv('../finalDataset.csv', index=False)

In [ ]:
valid_matches[['participantId','participantsAssisted','towerKillsAssisted','monsterKillsAssisted','participantsAssistedWithPressure']].head(5)